# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JAN-tech404/FlyRank.internee/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

I choose Lane 2: Refresh / Content Opportunity Scoring. This lane focuses on identifying which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring. This aligns with the internship goal of building decision-support tools that help content teams prioritize limited resources. By building a transparent scoring model with reason codes, we can provide actionable recommendations that are observable and defensible, avoiding the trap of merely copying existing product scores.

In [ ]:
# Quick look at the data: 2-3 real numbers to back lane choice
import pandas as pd

# Load the starter dataset
df = pd.read_csv('/home/jan/Git cloned repos/FlyRank.internee/data/raw/content_refresh_anonymized.csv')

# Basic counts
n_rows = len(df)
print(f"Total rows in starter dataset: {n_rows}")

# Decline label (as in starter label)
declining = df['trend_direction'] == 'down'
n_declining = declining.sum()
print(f"Number of declining pages (trend_direction == 'down'): {n_declining} ({n_declining/n_rows*100:.1f}%)")

# Average impressions for declining vs non-declining
avg_imp_decl = df.loc[declining, 'impressions_90d'].mean()
avg_imp_not = df.loc[~declining, 'impressions_90d'].mean()
print(f"Average impressions_90d for declining: {avg_imp_decl:.0f}")
print(f"Average impressions_90d for non-declining: {avg_imp_not:.0f}")

# Show a simple baseline score distribution (optional)
# We'll compute the starter baseline score as defined in the guide
visibility = df['ctr'] * df['impressions_90d']  # simplified
freshness = 1 / (1 + df['content_age_days'] / 365)  # inverse age
position = (21 - df['avg_position'].clip(1, 20)) / 20  # higher is better, cap at 20
depth = df['engagement_rate']  # placeholder
baseline = 0.4*visibility + 0.3*freshness + 0.25*position + 0.05*depth
print(f"Baseline score mean: {baseline.mean():.2f}, std: {baseline.std():.2f}")

## 2. The question: decision, action, cost of a wrong call

*Answer all four questions in writing.*

1. **What decision does this improve?**  
   Which content items should a content reviewer prioritize for review (refresh, expand, protect, prune, or monitor) given limited weekly capacity.

2. **Who acts on the output, and what do they do?**  
   A content editor or SEO specialist receives a ranked list of pages with reason codes and suggested actions. They review the top N pages each week, taking the suggested action (e.g., update content, improve metadata, expand coverage) or deciding to monitor further.

3. **What does a wrong answer cost?**  
   Reviewing a low‑opportunity page wastes the reviewer’s time that could have been spent on a higher‑impact page. If the reviewer can only review 20 pages per week, placing a low‑value page in the top 20 means a high‑value page is missed. Over a month, this could mean dozens of missed opportunities for traffic recovery or growth.

4. **Why does data or ML help at all?**  
   The relationship between historical signals (impressions, CTR, engagement, position, freshness, etc.) and future editorial need is complex and non‑linear. Simple rules (e.g., “if CTR < 0.5% and impressions > 500”) miss interactions and shifting baselines. A model can learn weighted combinations that better rank true opportunity while still being transparent via reason codes.

In [ ]:
# Code to back up section 2: numbers that illustrate decision impact
import pandas as pd
import numpy as np

df = pd.read_csv('/home/jan/Git cloned repos/FlyRank.internee/data/raw/content_refresh_anonymized.csv')

# Define declining label (as in starter)
declining = df['trend_direction'] == 'down'
print(f"Declining pages: {declining.sum()} / {len(df)} ({declining.mean()*100:.1f}%)")

# Simple signal: impressions_90d (higher = more opportunity)
# Rank by impressions descending
rank_by_imp = df['impressions_90d'].rank(ascending=False, method='first')
top20_by_imp = rank_by_imp <= 20  # top 20 rows
precision_at_20_imp = declining[top20_by_imp].sum() / 20
print(f"Precision@20 for declining pages when ranking by impressions_90d: {precision_at_20_imp:.2f}")

# Random baseline (expected precision = prevalence)
random_precision = declining.mean()
print(f"Expected precision@20 for random ranking: {random_precision:.2f}")

# Improvement factor
improvement = precision_at_20_imp / random_precision if random_precision > 0 else 0
print(f"Improvement over random: {improvement:.1f}x")

# What does a wrong recommendation cost? 
# If we can review 20 pages per week, using impressions ranking we catch about {precision_at_20_imp*20:.1f} declining pages in top 20.
# Random would catch only {random_precision*20:.1f} declining pages.
# So we gain about {(precision_at_20_imp - random_precision)*20:.1f} extra declining pages per week.
gain_per_week = (precision_at_20_imp - random_precision) * 20
print(f"Extra declining pages caught per week (vs random): {gain_per_week:.1f}")

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [ ]:
# Quick look at the data: 2-3 real numbers for Refresh lane
import pandas as pd

df = pd.read_csv('/home/jan/Git cloned repos/FlyRank.internee/data/raw/content_refresh_anonymized.csv')

# 1. Pages with low CTR (<0.5%) but high impressions (>=500) - CTR opportunity
low_ctr_high_imp = (df['ctr'] < 0.5) & (df['impressions_90d'] >= 500)
count_low_ctr_high_imp = low_ctr_high_imp.sum()
print(f"Pages with CTR < 0.5% and impressions_90d >= 500: {count_low_ctr_high_imp} ({count_low_ctr_high_imp/len(df)*100:.1f}%)")

# 2. Pages that are declining but still have decent impressions (demand exists)
declining = df['trend_direction'] == 'down'
declining_good_imp = declining & (df['impressions_90d'] >= 100)
count_declining_good_imp = declining_good_imp.sum()
print(f"Declining pages with impressions_90d >= 100: {count_declining_good_imp} ({count_declining_good_imp/len(df)*100:.1f}%)")

# 3. Average content age for declining vs non-declining pages
avg_age_decl = df.loc[declining, 'content_age_days'].mean()
avg_age_not = df.loc[~declining, 'content_age_days'].mean()
print(f"Average content age (days): declining={avg_age_decl:.1f}, non-declining={avg_age_not:.1f}")

# 4. Fraction of pages that are both stale (age >= 365) and have impressions >= 250 (thin+visible opportunity)
stale_visible = (df['content_age_days'] >= 365) & (df['impressions_90d'] >= 250)
count_stale_visible = stale_visible.sum()
print(f"Stale (>=365 days) and visible (impressions>=250): {count_stale_visible} ({count_stale_visible/len(df)*100:.1f}%)")

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim (observed/decision-support):**  
- I can observe which historical signals (impressions, CTR, engagement, position, freshness, etc.) are associated with pages that later decline or remain strong, using the defined future window.  
- I can produce a ranked list of pages where a reviewer should look first, based on a transparent score and reason codes that are computable before the decision point.  
- I can show that my ranking improves precision@K over a baseline rule (e.g., stale visible page) on historical validation, meaning a reviewer would find more actionable pages per unit effort.  
- I can state directional relationships: e.g., "pages with lower CTR and higher impressions tend to have higher opportunity scores" (an observed correlation, not a guarantee).

**What I cannot claim:**  
- I cannot prove that refreshing a specific page will cause traffic recovery (causal claim would require an experiment).  
- I cannot say that my score predicts Google's internal algorithm or future ranking changes; I only predict observable editorial need.  
- I cannot claim that my model uncovers the "true" reason for decline; I only surface correlates that are visible in the data.  
- I cannot generalize beyond the observed time window without assuming stationarity; I must validate on held-out time periods.

In [ ]:
# Quick validation: train a simple model and show it beats baseline on precision@20
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score

df = pd.read_csv('/home/jan/Git cloned repos/FlyRank.internee/data/raw/content_refresh_anonymized.csv')

# Features: select a few numeric columns
feature_cols = ['impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'content_age_days']
# Fill missing with 0 for simplicity
X = df[feature_cols].fillna(0)
# Label: declining (as proxy)
y = (df['trend_direction'] == 'down').astype(int)

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train logistic regression
clf = LogisticRegression(max_iter=1000, class_weight='balanced')
clf.fit(X_train, y_train)
y_pred_proba = clf.predict_proba(X_test)[:, 1]

# Get top 20% by predicted probability
top_n = int(len(y_test) * 0.2)
top_indices = np.argsort(y_pred_proba)[-top_n:]
y_top = y_test.iloc[top_indices]
# Precision@20%: fraction of positives in top 20%
precision_at_20 = y_top.mean()
print(f"Precision@20% (model): {precision_at_20:.3f}")

# Baseline: predict by prevalence (random)
baseline_prec = y_test.mean()
print(f"Baseline precision (random): {baseline_prec:.3f}")

# Improvement
improvement = (precision_at_20 - baseline_prec) / baseline_prec * 100
print(f"Improvement over baseline: {improvement:.1f}%")

# Also compute precision@20 for a simple rule: impressions_90d > median
median_imp = X['impressions_90d'].median()
rule_pred = (X['impressions_90d'] > median_imp).astype(int)
X_train_rule, X_test_rule, y_train_rule, y_test_rule = train_test_split(rule_pred, y, test_size=0.2, random_state=42)
# For rule, we need to get top 20% by rule score? Actually rule is binary; we can sort by the rule (1 then 0)
# We'll just compute precision if we take all predicted positives as top? Let's do: sort by rule_pred descending (so 1s first)
# Then take top 20% of samples
X_test_rule_series = pd.Series(X_test_rule, index=y_test.index)
ranked = X_test_rule_series.sort_values(ascending=False)
top_n_rule = int(len(y_test) * 0.2)
top_idx_rule = ranked.index[:top_n_rule]
precision_rule = y_test.loc[top_idx_rule].mean()
print(f"Precision@20% (rule: impressions > median): {precision_rule:.3f}")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.